# Lab 12: Faster R-CNN Inference

**Module 12** | Read `notes/12-faster-rcnn.pdf` before running this notebook.

Run a pre-trained Faster R-CNN on sample images, filter detections by score, and save annotated outputs.

Run every cell top to bottom. Code is complete, no fill-in sections. Markdown cells explain what each block does and why.


## Runtime device

PyTorch can run on your NVIDIA GPU through CUDA or fall back to CPU. GPU execution moves matrix operations off the host and typically speeds up neural network training by a large factor. The next cell detects what is available and prints the active device so you know whether to expect fast training or should keep batch sizes small.


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"CUDA enabled, {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("CUDA not available, using CPU. Labs still run; training may take longer.")


## Faster R-CNN object detection

Faster R-CNN is a two-stage detector: a backbone extracts features, a Region Proposal Network suggests candidate boxes, and a classification head refines each region. Torchvision ships a ResNet-50 FPN variant pre-trained on COCO.

This lab loads those weights, runs inference on local sample images, filters detections with confidence above 0.5, prints per-image detection details, builds a summary table, saves annotated copies, and previews one result inline.


### Load the pre-trained model

`FasterRCNN_ResNet50_FPN_Weights.DEFAULT` bundles both network weights and the preprocessing transforms (resize, normalize) expected at inference time. The weight metadata also includes the COCO category name list used to decode label indices.


In [ ]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from runtime_env import configure_runtime

configure_runtime(quiet=True)

import torch
from PIL import Image, ImageDraw, ImageFont
from torchvision.models.detection import (
    FasterRCNN_ResNet50_FPN_Weights,
    fasterrcnn_resnet50_fpn,
)
from runtime_env import ensure_sample_images

ROOT = Path("..").resolve()
IMAGE_DIR = ensure_sample_images()
OUT_DIR = ROOT / "labs" / "annotated"
OUT_DIR.mkdir(parents=True, exist_ok=True)

weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
model = fasterrcnn_resnet50_fpn(weights=weights)
model.eval().to(device)
preprocess = weights.transforms()
category_names = weights.meta["categories"]

image_paths = sorted(IMAGE_DIR.glob("*.jpg"))
print(f"Model: Faster R-CNN ResNet-50 FPN (COCO pre-trained)")
print(f"COCO categories: {len(category_names)}")
print(f"Sample images found: {len(image_paths)} in {IMAGE_DIR}")
if not image_paths:
    raise FileNotFoundError(
        f"No JPG files in {IMAGE_DIR}. Check your network connection or run: python download_datasets.py"
    )
print()
print("First 10 category names:", category_names[:10])


### Run inference and record detections

The model returns `boxes`, `labels`, and `scores` per image. We filter by `score > 0.5`, print each kept detection with class name, confidence, and box coordinates, draw rectangles on the image, and write PNG files to `labs/annotated/`.


In [ ]:
SCORE_THRESHOLD = 0.5
saved: list = []
summary_rows: list[dict] = []

try:
    font = ImageFont.truetype("arial.ttf", 16)
except OSError:
    font = ImageFont.load_default()

for img_path in image_paths:
    image = Image.open(img_path).convert("RGB")
    tensor = preprocess(image).to(device)
    with torch.no_grad():
        prediction = model([tensor])[0]

    draw = ImageDraw.Draw(image)
    detections: list[dict] = []

    print(f"\n{img_path.name} ({image.size[0]}x{image.size[1]})")
    print("-" * 70)

    for box, label, score in zip(
        prediction["boxes"], prediction["labels"], prediction["scores"]
    ):
        score_val = score.item()
        if score_val <= SCORE_THRESHOLD:
            continue
        x1, y1, x2, y2 = box.tolist()
        name = category_names[label.item()]
        detections.append({
            "class": name,
            "score": score_val,
            "box": (x1, y1, x2, y2),
        })
        print(
            f"  {name:20s}  score={score_val:.3f}  "
            f"box=[{x1:6.1f}, {y1:6.1f}, {x2:6.1f}, {y2:6.1f}]"
        )
        draw.rectangle([x1, y1, x2, y2], outline="lime", width=3)
        draw.text((x1, max(y1 - 18, 0)), f"{name} {score_val:.2f}", fill="yellow", font=font)

    if not detections:
        print("  (no detections above threshold)")

    out_path = OUT_DIR / f"{img_path.stem}_det.jpg"
    image.save(out_path)
    saved.append(out_path)

    classes_found = sorted({d["class"] for d in detections})
    summary_rows.append({
        "image": img_path.name,
        "detections": len(detections),
        "classes": ", ".join(classes_found) if classes_found else "(none)",
        "output": out_path.name,
    })

print(f"\nSaved {len(saved)} annotated images to {OUT_DIR}")


### Summary table across all images

This table gives a quick overview of how many objects each image triggered and which COCO classes appeared. Use it to spot images with many detections or unexpected class labels before opening the annotated files.


In [ ]:
print("Detection summary (score > {:.1f}):".format(SCORE_THRESHOLD))
print("=" * 90)
print(f"{'Image':<24} {'Count':>6}  {'Classes':<50} {'Output':<12}")
print("-" * 90)
for row in summary_rows:
    classes = row["classes"]
    if len(classes) > 48:
        classes = classes[:45] + "..."
    print(
        f"{row['image']:<24} {row['detections']:>6}  "
        f"{classes:<50} {row['output']:<12}"
    )
print("-" * 90)
total_dets = sum(r["detections"] for r in summary_rows)
print(f"Total detections kept: {total_dets} across {len(summary_rows)} images")


### Preview one result inline

Open the first annotated image in the notebook to verify labels and box placement visually.


In [ ]:
import matplotlib.pyplot as plt

preview = Image.open(saved[0])
plt.figure(figsize=(10, 7))
plt.imshow(preview)
plt.axis("off")
plt.title(f"{saved[0].name} (inline preview)")
plt.show()


### Evaluation

Check that the category count matches COCO (91 classes including background index 0), that per-image lists show plausible class names and scores, and that the summary table aligns with what you see in the inline preview. False positives often appear on cluttered backgrounds; raising `SCORE_THRESHOLD` reduces them at the cost of missed objects.


In [ ]:
all_classes = set()
for row in summary_rows:
    if row["classes"] != "(none)":
        all_classes.update(row["classes"].split(", "))

print("Evaluation checklist:")
print(f"  COCO categories loaded: {len(category_names)}")
print(f"  Images processed: {len(summary_rows)}")
print(f"  Unique classes detected: {len(all_classes)}")
if all_classes:
    print(f"  Classes seen: {sorted(all_classes)}")
print(f"  Annotated outputs in: {OUT_DIR}")
